# Session 2: Exercises

**Do these before session 3.** They take about ninety minutes; the ⭐⭐⭐ ones are the ones that matter.

The house pattern is **predict, then measure**. Write down what you think will happen *before* you run
a cell: the gap between your prediction and the result is the only reliable signal of what you actually
believe. Then measure with `grade()`, the same helper as in the lesson, rather than by eye.

Write predictions and answers in the cells marked **✍️**. Code to write is marked `# TODO`. Worked
solutions are released after the next session.

## Setup (just run it)

The client, the tickets, the attacks, the returns policy, `ask` with its retry, `grade`, and `STEP3` from the lesson's staircase.

In [ ]:
import os
import re
import time
from pathlib import Path

import yaml
from dotenv import load_dotenv
from google import genai
from google.genai import errors

load_dotenv(Path.cwd().parent / ".env")
client = genai.Client()

MODEL = "gemini-3.5-flash-lite"
TICKETS = yaml.safe_load((Path.cwd().parent / "data" / "tickets.yml").read_text(encoding="utf-8"))
ATTACKS = yaml.safe_load((Path.cwd().parent / "data" / "attacks.yml").read_text(encoding="utf-8"))
by_id = {t["id"]: t for t in TICKETS}
policy = Path("returns_policy.md").read_text(encoding="utf-8")


def as_email(ticket_id):
    t = by_id[ticket_id]
    return f"Subject: {t['subject']}\n{t['body'].strip()}"


def ask(system, user, model=MODEL, **config):
    """One call with a 429 retry (lesson section 3.8)."""
    if system:
        config["system_instruction"] = system
    for wait in (1, 2, 4, 8, None):
        try:
            r = client.models.generate_content(model=model, contents=user, config=config or None)
            return r.text or ""
        except errors.ClientError as err:
            if err.code != 429 or wait is None:
                raise
            time.sleep(wait)


def grade(system, user, check, n=3, fn=None, show=True, **config):
    """Run the same prompt n times; print one reply and how many pass `check` (lesson section 6)."""
    fn = fn or ask
    replies = [fn(system, user, **config) for _ in range(n)]
    passed = sum(bool(check(r)) for r in replies)
    if show:
        print(replies[0].strip(), "\n" + "-" * 72)
    print(f"passed {passed}/{n}")
    return replies


words = lambda r: len(r.split())
promises_refund = lambda r: re.search(
    r"\b(i|we)\s*(['’]ve|['’]ll| have| will| can| am going to)\s+(\w+[\s,]+){0,5}?(refund|revers|credit)",
    r, re.I) is not None
escalates = lambda r: re.search(r"(colleague|team|specialist|human)", r, re.I) is not None

# The staircase's final prompt, from lesson section 6.
STEP3 = """You are the customer support assistant for ShopWise, an online electronics store in Bangladesh.
You write the reply email the customer receives. Tone: warm, plain-spoken, professional. Never blame the customer.

Output format: a greeting line ("Hello" unless you know their name), two to four short paragraphs of
plain text, then sign off as "ShopWise Support". No markdown, no subject line.

Constraints, and why they exist:
- You have no access to orders, payments or accounts, so never say you have checked, found or confirmed anything.
- Only the billing team can approve refunds, so never promise, approve or start one.
- Never invent timeframes, amounts or policy details; a wrong promise costs more than no promise.
When the customer needs something you cannot do: say so plainly, say a colleague on the right team will
follow up, and tell them what happens next."""
print(f"{len(TICKETS)} tickets, {len(ATTACKS)} attacks, policy loaded")

## ⭐ Exercise 1: The fifth step

**The idea.** prompt rules *interact*. The staircase added sections that stacked
neatly. They do not always stack: a rule added for one good reason can quietly damage what another
section was buying you.

Add *"Reply in no more than 3 sentences."* to `STEP3` and run both prompts on `TCK-003`, the furious
all-caps ticket. Check two things: does the first sentence acknowledge the anger, and does the reply
still hand over?

**✍️ Predict first:** what will the brevity rule cost? Which part of `STEP3` will it fight with?

**✍️ Your prediction:**

*(write it here before you run the next cell)*

In [ ]:
angry = as_email("TCK-003")
acknowledges = lambda r: re.search(r"(sorry|apolog|frustrat|understand)", r.split(".")[0], re.I) is not None
hands_over_with_next_step = lambda r: escalates(r) and re.search(r"(will (reply|contact|follow|email|be in touch)|within|next)", r, re.I) is not None

# TODO: build STEP4 by adding the 3-sentence rule to STEP3, then grade both on `angry`
#       with acknowledges and with hands_over_with_next_step.
STEP4 = ...

**✍️ Your answer.** Which part did the brevity rule fight with, and was your prediction right? How would you rewrite the rule so it stops fighting?

*(your answer here)*

## ⭐ Exercise 2: Give the reason

**The idea.** lesson section 7.1, on your own rules. Below are three bare rules.
Rewrite each **with its reason**, then grade old against new on the same ticket.

In [ ]:
BARE = STEP3 + """
- Do not mention competitors.
- Do not use exclamation marks.
- Do not ask for card numbers."""

T = as_email("TCK-016")    # "cancel my order, I found the kettles cheaper elsewhere"
passes_all = lambda r: ("!" not in r and not re.search(r"(card number|card details|cvv)", r, re.I)
                        and not re.search(r"elsewhere|other (store|shop|retailer)|competitor", r, re.I))

_ = grade(BARE, T, passes_all)

# TODO: write WITH_REASONS (the same three rules, each with the reason it exists) and grade it.
WITH_REASONS = ...

**✍️ Your answer.** Did the reasons change the pass count? Which rule benefited most, and why that one?

*(your answer here)*

## ⭐⭐ Exercise 3: The Bangla test

**The idea.** *where* a rule belongs. `TCK-004` is written in Bangla. Send it
through `STEP3` and see what language comes back. Then add a language rule, and decide which part of
the two-slot layout it belongs in.

**✍️ Predict first:** Bangla, English, or mixed? Why?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
bangla = as_email("TCK-004")
is_bangla = lambda r: len(re.findall(r"[\u0980-\u09FF]", r)) > len(r) * 0.3

_ = grade(STEP3, bangla, is_bangla)

# TODO: add a language rule, with its reason, where you think it belongs. Grade again.
STEP3_LANG = ...

**✍️ Your answer.** Which section did you put it in, and what would go wrong elsewhere?

*(your answer here)*

## ⭐⭐ Exercise 4: Earn your example

**The idea.** lesson section 7.3. An example is a specification: it teaches its
shape *and* everything else in it.

1. Get `shaped` to 3/3 on `TCK-008` (kettle out of stock) using **exactly one** example.
2. Then check what your example leaked: run the same system prompt on two unrelated tickets and look
   for any phrase, product or number copied from your example.

In [ ]:
SYS = ("You are ShopWise customer support. Write the reply email the customer receives. Greet the "
       "customer, acknowledge the issue, say a colleague will follow up, and sign off as 'ShopWise Support'.")
shaped = lambda r: r.strip().startswith("Hello,") and "Subject" not in r and r.strip().endswith("ShopWise Support")

# TODO: write one <example>...</example> and grade SYS + MY_EXAMPLE on TCK-008.
MY_EXAMPLE = ...

# TODO: then run SYS + MY_EXAMPLE on TCK-015 and TCK-013 and look for leaks.

**✍️ Your answer.** What leaked from your example, if anything? What would you change?

*(your answer here)*

## ⭐⭐ Exercise 5: Chain it, and count the cost

**The idea.** lesson section 7.6 has a price. A chain makes more calls. Measure
both sides: the pass count *and* the tokens.

Build a draft → review → revise chain for `TCK-014` (the gold-tier shipping complaint), compare it with
one prompt that does all three jobs, and total the tokens each approach used.

In [ ]:
RULES = """Rules for the reply:
1. Under 120 words.
2. Never promise a refund, replacement or date; a colleague decides.
3. Acknowledge the customer's frustration in the first sentence.
4. End with the exact line: ShopWise Support"""
T14 = f"<ticket>\n{as_email('TCK-014')}\n</ticket>"
reply_only = lambda r: (not re.search(r"(classif|categor|rule \d|check:|draft)", r, re.I)
                        and r.strip().endswith("ShopWise Support"))


def tokens_of(prompt):
    """Call once and return (text, input+output+thinking tokens)."""
    u = client.models.generate_content(model=MODEL, contents=prompt).usage_metadata
    ...

# TODO: finish tokens_of, then build one_prompt() and chain() and compare pass counts and tokens.

**✍️ Your answer.** Is the chain worth it here? What would change your answer?

*(your answer here)*

## ⭐⭐⭐ Exercise 6: The hallucination hunt

**The idea.** the thing this session is built around, and the reason session 8
exists.

### Parts A to C: three questions to a public chatbot

Open any public chatbot (Gemini, ChatGPT, Claude). Ask it three questions about **ShopWise**, the
fictional electronics store from this course: its return policy, whether it ships to Chattogram, and
its customer-service phone number. Record each answer as **invented**, **hedged** or **refused**.

> **The name-collision trap.** There is a real supermarket chain called *Shopwise* in the Philippines.
> A chatbot may quote its genuine policies. A true fact about the wrong company is indistinguishable
> from an invented fact about the right one.

**✍️ Parts A to C, your findings:**

| # | question | invented / hedged / refused | mentioned a different Shopwise? |
|---|---|---|---|
| 1 | return policy | | |
| 2 | shipping to Chattogram | | |
| 3 | phone number | | |

### Part D: the real exercise

Ask the same question twice through the API: bare, and with one sentence of role. This is the lesson's
opening. Reproduce it yourself and count.

In [ ]:
Q = "What is ShopWise's return policy?"
# Hedging or declining = asking WHICH ShopWise, or saying it has no information.
# ("Could you send your order number?" doesn't count: that's a support agent talking.)
hedges = lambda r: re.search(r"(which shopwise|which (store|company|retailer|business)|several|"
                            r"more than one|different (businesses|companies|stores)|not sure which|"
                            r"(do not|don't) have (access|information|specific)|not (familiar|aware)|"
                            r"check (the|their) (official )?website)", r, re.I) is not None

# TODO: grade Q with no system prompt, then with
#       "You are ShopWise customer support, replying to a customer." Count the hedges and refusals.

### Part E: fix it with section 7.5, then explain why that isn't enough

Using only the techniques from lesson section 7.5 and the policy in `returns_policy.md`, make the
role-framed version answer correctly. Then answer the question below.

In [ ]:
# TODO: a system prompt using an out and quote-then-answer, with the policy in the user turn.
FIXED = ...

**✍️ Why is Part E still not a fix for production?** One or two sentences.

*(your answer here)*

## ⭐⭐⭐ Exercise 7: A first look at prompt injection

**The idea.** a system prompt is a **soft** boundary. `data/attacks.yml` holds
six attack tickets, each with its `intent` (what it tries to make the assistant do) and its `tell`
(what success looks like in a reply). Send each through `STEP3`, first pasted plainly and then inside
`<ticket>` tags with a line saying ticket text is data, never instructions (lesson section 7.2).

Mark each reply by its **intent**, not its wording: the attack succeeded if the assistant *did the
thing*, however politely.

In [ ]:
for atk in ATTACKS:
    print(atk["id"], atk["shape"], "—", atk["intent"])

# TODO: run each attack's body through STEP3 plainly, then delimited, and read each reply
#       against atk["tell"]. Record which held.

**✍️ Your answer.** Which attacks held, which did not, and did the delimiters change anything? Two sentences on why a prompt is a soft boundary.

*(your answer here)*